# Restroom Icon Matching

In [1]:
import random
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
import os
from PIL import Image
import torch.nn.functional as F
import torchvision 
import torchvision.models as models
import albumentations as A
import torch.nn as nn 
from sklearn.neighbors import NearestNeighbors 
from sklearn.preprocessing import normalize 
import timm 

seed = 42

random.seed(seed)    
np.random.seed(seed)   
torch.manual_seed(seed)    
torch.cuda.manual_seed(seed)       
torch.cuda.manual_seed_all(seed)   

# Ensures deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [2]:
def sort_paths_by_number(path_list):
    """
    Sort based on the numerical values of the filenames in the path,
    assuming all filenames can be converted to integers.
    """
    def get_file_number(path):
        file_name = os.path.splitext(os.path.basename(path))[0]
        return int(file_name)

    path_list.sort(key=get_file_number)

In [3]:
DEVICE = 'cuda'
model_name = 'swinv2_large_window12to16_192to256'
model = timm.create_model(model_name, pretrained=True, num_classes=0)
model = nn.DataParallel(model.to(DEVICE))
for param in model.parameters() : 
    param.requires_grad = False
model.eval()

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

DataParallel(
  (module): SwinTransformerV2(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 192, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((192,), eps=1e-05, elementwise_affine=True)
    )
    (layers): Sequential(
      (0): SwinTransformerV2Stage(
        (downsample): Identity()
        (blocks): ModuleList(
          (0): SwinTransformerV2Block(
            (attn): WindowAttention(
              (cpb_mlp): Sequential(
                (0): Linear(in_features=2, out_features=512, bias=True)
                (1): ReLU(inplace=True)
                (2): Linear(in_features=512, out_features=6, bias=False)
              )
              (qkv): Linear(in_features=192, out_features=576, bias=False)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=192, out_features=192, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (norm1): LayerNor

In [4]:
preprocess = A.Compose([
    A.Resize(256, 256), 
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    A.ToTensorV2(), 
])

In [5]:
def infer(img_paths):
    """
    Compute L2‑normalized feature embeddings for a list of image file paths using the CLIP visual encoder.
    """
    embeddings = []
    for path in tqdm(img_paths):
        img = np.array(Image.open(path).convert('RGB'))
        x = preprocess(image=img)['image']
        x = x.unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            emb = model(x)

        embeddings.append(emb)

    embeddings = torch.cat(embeddings)
    embeddings = F.normalize(embeddings, p=2, dim=1)
    
    return embeddings.cpu().detach().numpy()

In [6]:
def match_images(BASE_DATA_DIR):
    """
    For each query image in BASE_DATA_DIR/query, find its best matching image
    in BASE_DATA_DIR/gallery by computing cosine similarity of CLIP embeddings,
    then return the match indices as a list.
    """
    QUERY_DIR = BASE_DATA_DIR / "query"
    NON_QUERY_DIR = BASE_DATA_DIR / "gallery"

    query_image_paths = list(QUERY_DIR.glob("*.png"))
    non_query_image_paths = list(NON_QUERY_DIR.glob("*.png"))

    query_image_paths_str = [str(p) for p in query_image_paths]
    non_query_image_paths_str = [str(p) for p in non_query_image_paths]

    sort_paths_by_number(query_image_paths_str)
    sort_paths_by_number(non_query_image_paths_str)

    query_embeddings = infer(query_image_paths_str)
    non_query_embeddings = infer(non_query_image_paths_str)

    knn = NearestNeighbors(n_neighbors=2, metric='cosine')
    knn.fit(non_query_embeddings)
    matches_idxs = knn.kneighbors(query_embeddings, n_neighbors=2, return_distance=False)
    idxs = []
    for i in range(len(matches_idxs)):
        idxs.append(matches_idxs[i][1]+1)

    return idxs

In [7]:
import pandas as pd
from pathlib import Path

# Assuming match_images returns a list of matches, for example:
# test_results = [(1, 2), (3, 4), (5, 6)] or test_results = [1, 2, 3, 4]

DATA_PATH = Path("/kaggle/input/competitions/restroom-ioai-2025/test_set")

print("Processing test set...")
test_results = match_images(DATA_PATH / "test_set")

test_results = [int(x) for x in test_results]

submission_df = pd.DataFrame({
    'id': [0],
    'array': [test_results] # Use the clean string, NOT str(list)
})

# Save to CSV
output_file = "Restroom_Baseline.csv"
submission_df.to_csv(output_file, index=False)

print(f"Submission file created successfully at {output_file}")

Processing test set...


100%|██████████| 80/80 [00:05<00:00, 15.07it/s]

Submission file created successfully at Restroom_Baseline.csv


In [8]:
submission_df['array'][0]

[37,
 53,
 45,
 17,
 58,
 11,
 14,
 59,
 39,
 7,
 18,
 21,
 44,
 15,
 22,
 19,
 57,
 35,
 27,
 6,
 50,
 12,
 16,
 9,
 54,
 31,
 20,
 29,
 15,
 28,
 62,
 78,
 74,
 79,
 63,
 65,
 75,
 71,
 73,
 80]